# Biblical Gemma 4 31B Fine-Tuning with Unsloth (4-bit QLoRA)

**Base Model:** Gemma 4 31B Instruct

**Dataset:** Per-persona datagen JSONL — each persona has its own system prompt and distinctive voice

**Training Hardware:** NVIDIA DGX Spark (128GB unified memory)

**Deployment Target:** Standard deployment profile (persona-aware base LoRA for production use)

**Chat Template:** Tokenizer-native Gemma chat template (applied via `tokenizer.apply_chat_template`)

**Architecture:** This base LoRA teaches the model persona-switching — "when the system prompt says you're Amos, speak like Amos; when it says David, speak like David." Optional persona LoRAs can refine individual voices further.

## 1. Configuration

All paths and variables for easy configuration.

In [ ]:

# =========================== PATHS (all cascade from PROJECT_ROOT) ===========================
PROJECT_ROOT = "/workspace/training/biblical"
OUTPUT_ROOT = f"{PROJECT_ROOT}/output"

# =========================== MODEL CONFIGURATION ===========================
# unsloth/gemma-4-12b-it has no dedicated pre-quantized bnb-4bit repo on HF;
# Unsloth applies bnb 4-bit quantization on the fly via load_in_4bit=True below.
BASE_LLM = "unsloth/gemma-4-31B-it"
MODEL_NAME_BASE = "biblical_gemma4_31b_unsloth_4bit"

# =========================== INPUT DATA ===========================
# Combined multi-turn ShareGPT JSONL from datagen notebooks (per-persona + augmented data merged).
# Already quality-filtered, multi-turn (4 QA pairs per conversation), grouped by topic
INPUT_DATA_FILE = f"{PROJECT_ROOT}/data/training-data/biblical_persona_v2/biblical_personas_combined_sharegpt.jsonl"

# =========================== PERSONA SYSTEM PROMPTS ===========================
# System prompts are EXTRACTED from the JSONL at load time (see data loading cell below).
# This keeps training in sync with datagen — if you regenerate data with new/changed
# prompts, the training notebook picks them up automatically.
# After loading, the dict `persona_system_prompts` maps persona_key -> full prompt text.
# It is also saved alongside the LoRA adapters for use at inference time.

# =========================== OUTPUT DIRECTORIES ===========================
OUTPUT_BASE_DIR = f"{OUTPUT_ROOT}/{MODEL_NAME_BASE}"
OUTPUT_DIR_ADAPTERS = f"{OUTPUT_BASE_DIR}/train"
LORA_OUTPUT_DIR = f"{OUTPUT_BASE_DIR}/lora_adapters"

# =========================== TRAINING HYPERPARAMETERS ===========================
MAX_SEQ_LENGTH = 4096
BATCH_SIZE = 2
GRAD_ACCUM = 4
LEARNING_RATE = 2e-4
TARGET_EPOCHS = 1

# =========================== LoRA CONFIGURATION ===========================
# Adapter is merged into the base weights at GGUF export, so adapter size on
# disk is irrelevant. Use full attention + MLP targets and a higher rank for
# stronger persona-distinction learning.
LORA_R = 32
LORA_ALPHA = 32
LORA_DROPOUT = 0

# Full "all-linear" Unsloth recipe: attention + MLP projections.
LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

# =========================== INFERENCE TEST ===========================
TEST_PROMPT = "I am struggling with forgiveness. What does Scripture teach about forgiving others?"

# ============================================================================
print("✓ Configuration loaded (Gemma 4 12B 4-bit QLoRA)"  )
print(f"  Base model: {BASE_LLM}")
print(f"  Model name: {MODEL_NAME_BASE}")
print(f"  Input data: {INPUT_DATA_FILE}")
print(f"  Output base: {OUTPUT_BASE_DIR}")
print(f"  LoRA output: {LORA_OUTPUT_DIR}")
print(f"  LoRA config: r={LORA_R}, alpha={LORA_ALPHA}, targets={LORA_TARGET_MODULES}")
print(f"  Training: batch={BATCH_SIZE}, grad_accum={GRAD_ACCUM}, lr={LEARNING_RATE}")
print(f"  Training precision: 4-bit QLoRA")
print(f"  Max seq length: {MAX_SEQ_LENGTH}")
print(f"  Persona prompts: extracted from JSONL at load time")


✓ Configuration loaded (Gemma 4 12B 4-bit QLoRA)
  Base model: unsloth/gemma-4-31b-it
  Model name: biblical_gemma4_31b_unsloth_4bit
  Input data: /workspace/training/biblical/data/training-data/biblical_persona_v2/biblical_personas_combined_sharegpt.jsonl
  Output base: /workspace/training/biblical/output/biblical_gemma4_31b_unsloth_4bit
  LoRA output: /workspace/training/biblical/output/biblical_gemma4_31b_unsloth_4bit/lora_adapters
  LoRA config: r=32, alpha=32, targets=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
  Training: batch=2, grad_accum=4, lr=0.0002
  Training precision: 4-bit QLoRA
  Max seq length: 4096
  Persona prompts: extracted from JSONL at load time


## 2. Environment Preparation

Install Unsloth and updated HuggingFace libraries.

In [2]:
# Install core packages in the running notebook container
!pip install -q -U unsloth trl accelerate datasets bitsandbytes

# Container ships torchao 0.14.0+git (custom aarch64 build). peft requires
# torchao>=0.16.0 OR torchao absent. No aarch64 wheel ≥0.16 on PyPI, so
# uninstall — peft's torchao dispatcher then no-ops and falls through to
# the bnb 4-bit dispatcher, which is what we want for QLoRA anyway.
!pip uninstall -y -q torchao

# PIN: Gemma 4 31B uses the `gemma4` arch. transformers >= 5.14.0 ships the
# heterogeneous per-layer config, which rewrites `global_head_dim` into a
# per-layer override of `head_dim`. Unsloth reads it with a plain
# getattr(config, "head_dim") during its flash-attention probe, which then
# raises AmbiguousGlobalPerLayerAttributeError before any weights load.
# 5.13.1 is the last release that has the gemma4 model and NOT the mixin.
!pip install -q "transformers==5.13.1"

# Keep PEFT compatible with latest Transformers main.
!pip install -q -U git+https://github.com/huggingface/peft.git

# Verify installations
import importlib.util
import unsloth
import transformers
import peft
import trl
print(f"✓ Unsloth: {unsloth.__version__}")
print(f"✓ Transformers: {transformers.__version__}")
print(f"✓ PEFT: {peft.__version__}")
print(f"✓ TRL: {trl.__version__}")
print(f"✓ torchao installed: {importlib.util.find_spec('torchao') is not None} (should be False)")
print("Environment ready. Restart kernel, then rerun from Cell 3.")


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mergekit 0.1.4 requires accelerate~=1.6.0, but you have accelerate 1.14.0 which is incompatible.
mergekit 0.1.4 requires safetensors~=0.5.2, but you have safetensors 0.8.0 which is incompatible.
  DEPRECATION: Setting PIP_CONSTRAINT will not affect build constraints in the future, pip 26.2 will enforce this behaviour change. A possible replacement is to specify build constraints using --build-constraint or PIP_BUILD_CONSTRAINT. To disable this warning without any build constraints set --use-feature=build-constraint or PIP_USE_FEATURE="build-constraint".
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth-zoo 2026.7.6 requires torchao>=0.13.0; sys_platform != "darwin" or platform_machin

## 3. Load Dataset

Load the combined multi-turn ShareGPT JSONL from `biblical_datagen.ipynb`.

- Already quality-filtered (no short answers, no AI refusals)
- Multi-turn: 4 QA pairs grouped per conversation by topic
- Each conversation has a persona-specific system prompt
- Standard ShareGPT format: `[system, human, gpt, human, gpt, ...]`

In [3]:

import json, os, re
from collections import defaultdict
from datasets import Dataset as HFDataset

print(f"LOADING COMBINED SHAREGPT DATA")
print(f"  File: {INPUT_DATA_FILE}")

# Load multi-turn conversations and EXTRACT system prompts from the JSONL.
# This replaces hardcoded prompt dicts — prompts stay in sync with datagen automatically.
conversations = []
persona_system_prompts = {}   # persona_key -> full system prompt text
persona_counts = defaultdict(int)

with open(INPUT_DATA_FILE) as f:
    for line in f:
        conv = json.loads(line)
        conversations.append(conv)

        # Extract persona name from "You are <Name>, ..." pattern
        sys_msg = conv["conversations"][0]["value"]
        match = re.match(r"You are (.+?),", sys_msg)
        if match:
            raw_name = match.group(1)
            # Normalize to snake_case key: lowercase, strip leading "the ", underscores for spaces
            key = raw_name.lower()
            key = re.sub(r"^the\s+", "", key)
            key = key.replace(" ", "_")
            persona_counts[key] += 1
            if key not in persona_system_prompts:
                persona_system_prompts[key] = sys_msg
        else:
            print(f"  ⚠️ Could not extract persona from system prompt: {sys_msg[:80]}...")

dataset = HFDataset.from_list(conversations)

print(f"\n{'='*50}")
print(f"Total dataset: {len(dataset)} multi-turn conversations across {len(persona_counts)} personas")
print(f"Extracted {len(persona_system_prompts)} unique system prompts from JSONL")
print(f"Columns: {dataset.column_names}")
print(f"\nPer-persona breakdown:")
for p, c in sorted(persona_counts.items(), key=lambda x: -x[1]):
    print(f"  {p:20s} {c:>5d} conversations")

# Show a sample prompt to verify extraction
sample_key = next(iter(persona_system_prompts))
print(f"\n--- Sample extracted prompt ({sample_key}, first 200 chars) ---")
print(f"  {persona_system_prompts[sample_key][:200]}...")


LOADING COMBINED SHAREGPT DATA
  File: /workspace/training/biblical/data/training-data/biblical_persona_v2/biblical_personas_combined_sharegpt.jsonl

Total dataset: 4352 multi-turn conversations across 26 personas
Extracted 26 unique system prompts from JSONL
Columns: ['conversations', 'data_type']

Per-persona breakdown:
  moses                  785 conversations
  jeremiah               381 conversations
  paul                   377 conversations
  david                  350 conversations
  ezekiel                341 conversations
  isaiah                 332 conversations
  solomon                254 conversations
  job                    244 conversations
  daniel                 239 conversations
  peter                  148 conversations
  zechariah              145 conversations
  hosea                  107 conversations
  amos                    92 conversations
  joshua                  88 conversations
  micah                   73 conversations
  apostle_john            71 co

## 4. Validate & Summarize Dataset

Datagen data is already clean (no artifacts to strip). Verify data quality and show persona distribution.

In [4]:

bad_examples = []
empty_responses = []
unique_system_prompts = set()

for i, example in enumerate(dataset):
    convs = example["conversations"]
    # Multi-turn ShareGPT: system, then alternating human/gpt pairs
    if len(convs) < 3 or len(convs) % 2 == 0:
        bad_examples.append((i, f"Expected odd turn count ≥3, got {len(convs)}"))
        continue
    if convs[0]["from"] != "system":
        bad_examples.append((i, f"First turn should be 'system', got '{convs[0]['from']}'"))
        continue
    # Validate alternating human/gpt after system
    role_ok = True
    for j in range(1, len(convs)):
        expected = "human" if j % 2 == 1 else "gpt"
        if convs[j]["from"] != expected:
            bad_examples.append((i, f"Turn {j} should be '{expected}', got '{convs[j]['from']}'"))
            role_ok = False
            break
    if not role_ok:
        continue
    # Check last GPT response is not empty
    if len(convs[-1]["value"].strip()) == 0:
        empty_responses.append(i)
    unique_system_prompts.add(convs[0]["value"])

# Turn-count distribution
from collections import Counter
turn_dist = Counter(len(ex["conversations"]) for ex in dataset)

print("DATA QUALITY CHECK")
print(f"  Total examples: {len(dataset)}")
print(f"  Bad structure: {len(bad_examples)}")
print(f"  Empty responses: {len(empty_responses)}")
print(f"  Unique system prompts: {len(unique_system_prompts)} (should match extracted count: {len(persona_system_prompts)})")
print(f"  Turn distribution: {dict(sorted(turn_dist.items()))}")

if bad_examples:
    print(f"\n⚠️ Bad examples (first 5):")
    for idx, reason in bad_examples[:5]:
        print(f"    Example {idx}: {reason}")

if empty_responses:
    print(f"\n⚠️ Filtering {len(empty_responses)} empty responses...")
    good_indices = [i for i in range(len(dataset)) if i not in set(empty_responses)]
    dataset = dataset.select(good_indices)
    print(f"  Dataset after filtering: {len(dataset)} examples")

# Persona distribution
print(f"\nPERSONA DISTRIBUTION:")
max_name_len = max(len(n) for n in persona_counts)
for name, count in sorted(persona_counts.items(), key=lambda x: -x[1]):
    bar = "█" * (count // 50) + "▌" * (1 if count % 50 >= 25 else 0)
    print(f"  {name:<{max_name_len}} {count:>5}  {bar}")
print(f"  {'TOTAL':<{max_name_len}} {sum(persona_counts.values()):>5}")

# Show voice differentiation — first response from different personas
print(f"\nVOICE SAMPLES (first ~100 chars of response):")
seen_personas = set()
for example in dataset:
    system = example["conversations"][0]["value"]
    # Extract persona name from system prompt "You are X, ..."
    name_part = system.split(",")[0].replace("You are ", "")
    if name_part not in seen_personas and len(seen_personas) < 4:
        response_start = example["conversations"][2]["value"][:100]
        print(f"  {name_part}: \"{response_start}...\"")
        seen_personas.add(name_part)

print(f"\n✓ Dataset validated and ready for training")


DATA QUALITY CHECK
  Total examples: 4352
  Bad structure: 0
  Empty responses: 0
  Unique system prompts: 26 (should match extracted count: 26)
  Turn distribution: {3: 1741, 5: 91, 7: 541, 9: 1979}

PERSONA DISTRIBUTION:
  moses          785  ███████████████▌
  jeremiah       381  ███████▌
  paul           377  ███████▌
  david          350  ███████
  ezekiel        341  ██████▌
  isaiah         332  ██████▌
  solomon        254  █████
  job            244  ████▌
  daniel         239  ████▌
  peter          148  ██▌
  zechariah      145  ██▌
  hosea          107  ██
  amos            92  █▌
  joshua          88  █▌
  micah           73  █
  apostle_john    71  █
  james           54  █
  malachi         44  ▌
  joel            44  ▌
  zephaniah       37  ▌
  habakkuk        32  ▌
  jonah           29  ▌
  nahum           27  ▌
  haggai          24  
  obadiah         19  
  jude            15  
  TOTAL         4352

VOICE SAMPLES (first ~100 chars of response):
  Daniel: "Four is the

## 5. Load Model & Tokenizer (4-bit)

Load Gemma 4 12B in 4-bit precision for QLoRA training.

- **DGX Spark (128GB):** ample headroom for training and experimentation
- **Standard deployment:** production-ready persona-aware LoRA for use at scale

In [5]:
import os
# DGX Spark (sm_120 / GB10): disable Unsloth's flex_attention override and broken
# torch.compile path. Without these, Gemma falls back to slow eager Python loops.
# Must be set BEFORE `import unsloth`.
os.environ["UNSLOTH_ENABLE_FLEX_ATTENTION"] = "0"
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"

from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_LLM,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)


# ---- Multimodal base: unwrap the Processor -------------------------------------
# This base is a *ForConditionalGeneration checkpoint, so Unsloth loads it via
# AutoModelForVision2Seq and returns a multimodal Processor, not a tokenizer.
# Unwrap it and use the inner tokenizer for the rest of the notebook:
#   - TRL takes its text path instead of calling process_row(), which expects images.
#   - Trainer._get_train_sampler reads processing_class.model_input_names[0]; on a
#     Processor that is "pixel_values", not "input_ids".
#   - A Processor's first positional parameter is `images`, so tokenizer("text") binds
#     to the wrong argument - always call it as tokenizer(text=...).
# `processor` is kept so the adapter directory is saved with the same files as before.
processor = None
if hasattr(tokenizer, "tokenizer"):
    processor = tokenizer
    tokenizer = processor.tokenizer
    print("  (Extracted tokenizer from Processor - text-only SFT mode)")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id
# Gemma 4 can return a Processor object instead of a plain tokenizer.
if hasattr(tokenizer, "vocab_size"):
    vocab_size = tokenizer.vocab_size
elif hasattr(tokenizer, "tokenizer") and hasattr(tokenizer.tokenizer, "vocab_size"):
    vocab_size = tokenizer.tokenizer.vocab_size
else:
    vocab_size = "unknown"

print(f"✓ Model loaded: {BASE_LLM}")
print(f"  Precision: 4-bit (QLoRA)")
print(f"  Max sequence length: {MAX_SEQ_LENGTH}")
print(f"  Vocab size: {vocab_size}")
print(f"  Attn impl: {getattr(model.config, '_attn_implementation', 'unknown')}")

==((====))==  Unsloth 2026.7.5: Fast Gemma4 patching. Transformers: 5.15.0.dev0.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 121.689 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0a0+b558c986e8.nv25.11. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33+aa7bc36.d20260302. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


AmbiguousGlobalPerLayerAttributeError: 'head_dim' is a per-layer attribute and may vary across layers. Access it via the individual layer configs instead (e.g. config.per_layer_config[i].head_dim). To read the global config value from config.head_dim anyway, set `allow_global_per_layer_attribute_access` to `True` on the config. Warning: only do this if the caller can safely handle heterogeneous configs; code that assumes a homogeneous model may use the global value incorrectly.

## 6. Format Dataset for Chat Template

Apply Gemma's tokenizer chat template to each conversation and build the final training dataset.

In [ ]:
# Standardize ShareGPT rows, then format with Gemma tokenizer chat template
from unsloth.chat_templates import standardize_sharegpt

dataset = standardize_sharegpt(dataset)
formatted_texts = tokenizer.apply_chat_template(
    list(dataset["conversations"]),
    tokenize=False,
)

# Build final dataset
import pandas as pd
from datasets import Dataset as HFDataset
dataset = HFDataset.from_pandas(pd.DataFrame({"text": formatted_texts}))

# Filter out empty examples and shuffle
dataset = dataset.filter(lambda x: len(x["text"]) > 0)
dataset = dataset.shuffle(seed=42)

print(f"--- Sample formatted text (first 500 chars) ---")
print(dataset[0]['text'][:500])
print(f"\n\u2713 Dataset formatted: {len(dataset)} examples")

## 7. Add LoRA Adapters

Configure LoRA for efficient fine-tuning. See Step 1 config for module options.

### Freezing the vision / audio towers — deliberately, not by accident

`unsloth/gemma-4-31b-it` is a multimodal checkpoint: alongside the language model it carries BOTH a vision tower and an audio tower. This notebook trains a **text-only** persona LoRA, so none of that should receive an adapter.

That does **not** happen for free. PEFT matches a `target_modules` list like `["q_proj", "k_proj", ...]` by **suffix, across the entire module tree** — it has no notion of "the language model". Whether a tower gets caught depends purely on whether it reuses those leaf names:

- **Gemma 4** towers use `self_attn.{q,k,v,o}_proj` and `mlp.{gate,up,down}_proj` — identical to the language model, so a bare list catches every one of them.
- **Qwen VL** towers use `attn.qkv` / `attn.proj` / `linear_fc1` — no collision, so they escape. By luck of naming, not by design.

Two adapters in this workspace shipped with 224 vision and 72 audio LoRA tensors attached this way. They were harmless in effect — `lora_B` is zero-initialised and text batches never invoke those towers, so every leaked tensor stayed exactly zero — but they were 36–42% dead weight, and **vLLM refuses to load an adapter carrying vision-layer LoRA**.

The `finetune_*` flags below make the scoping explicit, and the assertion after the call proves it held rather than assuming it.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=LORA_TARGET_MODULES,
    # ---- TOWER FREEZE: deliberate, not incidental ----------------------------------
    # These four flags are load-bearing. Unsloth routes an explicit target_modules LIST
    # through its language-scoped regex ONLY when at least one of them is False; all four
    # default to True, so a bare list goes straight to PEFT, which matches by SUFFIX
    # across the WHOLE module tree. finetune_vision_layers=False is what confines the
    # match to the language model - it excludes the audio tower and MTP head too, since
    # the scoped regex requires a `language`/`text` path segment.
    finetune_vision_layers=False,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    # Deliberately True, NOT "unsloth". VERIFIED in
    # unsloth_zoo/gradient_checkpointing.py: the "unsloth" path offloads saved
    # activations to PINNED host memory (torch.empty(..., device="cpu",
    # pin_memory=True)) and grows those buffers with `if new_size > x.numel():
    # x.resize_(new_size)` - up only. The sole shrink path is
    # unpatch_unsloth_smart_gradient_checkpointing(), which runs at teardown, never
    # mid-run. On a discrete GPU that trade is a win: VRAM is scarce, host RAM is not.
    # On GB10 host and device are the SAME 128 GB pool, so the copy frees nothing and
    # adds a second, page-locked, monotonically growing copy the OS cannot reclaim.
    use_gradient_checkpointing=True,
    random_state=3407,
    max_seq_length=MAX_SEQ_LENGTH
)

print(f"\u2713 LoRA adapters added (r={LORA_R}, targets={LORA_TARGET_MODULES})")
print(f"\u2713 Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# ============ VERIFY THE ADAPTER SCOPE AND THE TOWER FREEZE ============
# This is a TEXT-ONLY fine-tune on a multimodal base. Two properties must hold, and they
# fail in different ways, so both are checked:
#
#   1. WHERE ADAPTERS ATTACHED. PEFT matches a target_modules LIST by suffix across the
#      whole module tree - it has no concept of "the language model". Any tower reusing
#      the language model's leaf names (q_proj, gate_proj, ...) collects adapters too.
#   2. WHAT IS TRAINABLE. Under QLoRA base weights are frozen 4-bit and only LoRA A/B
#      carry gradients, so the towers are frozen by construction. Assert it anyway - an
#      assumption that is never checked is an assumption that eventually breaks.
#
# See docs/multimodal_and_hybrid_base_models.md.
from collections import Counter as _Counter


def _zone_of(param_name):
    """Return the non-language sub-model a parameter belongs to, or None."""
    n = param_name.replace("base_model.model.", "", 1)
    if n.startswith("mtp."):
        return "mtp head"
    if "vision_tower" in n or ".visual." in n or n.startswith("visual."):
        return "vision tower"
    if "audio_tower" in n or n.startswith("audio_tower."):
        return "audio tower"
    if "multi_modal_projector" in n or "embed_vision" in n or "embed_audio" in n:
        return "mm projector"
    return None


_adapted = sorted({
    n.split(".lora_A")[0].split(".lora_B")[0].replace("base_model.model.", "")
    for n, _ in model.named_parameters() if ".lora_A." in n or ".lora_B." in n
})
_stray = [n for n in _adapted if _zone_of(n)]
if _stray:
    raise RuntimeError(
        f"LoRA attached to {len(_stray)} module(s) outside the language model "
        f"(e.g. {_stray[:3]}). The finetune_* scoping flags did not take effect. "
        "Do not train - vLLM refuses adapters carrying vision-layer LoRA."
    )

_unfrozen, _zone_params = {}, _Counter()
for _n, _p in model.named_parameters():
    _z = _zone_of(_n)
    if _z is None:
        continue
    _zone_params[_z] += 1
    if _p.requires_grad:
        _unfrozen.setdefault(_z, []).append(_n)

print(f"  LoRA adapted {len(_adapted)} modules, all inside the language model.")
for _z, _c in sorted(_zone_params.items()):
    _bad = len(_unfrozen.get(_z, []))
    print(f"  {_z:<14} {_c:>5} params  "
          f"{'FROZEN' if not _bad else str(_bad) + ' TRAINABLE - BAD'}")
if _unfrozen:
    raise RuntimeError(
        f"{sum(len(v) for v in _unfrozen.values())} tower/MTP parameter(s) are trainable "
        f"(e.g. {[n for v in _unfrozen.values() for n in v][:3]}). This is a text-only "
        "fine-tune - they must stay frozen."
    )


## 8. Trainer Setup

- 1 epoch, low learning rate (1e-4) — gently teaches persona-switching without degrading base capabilities
- Each example has a persona-specific system prompt so the model learns distinct voices

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        max_length=MAX_SEQ_LENGTH,
        # packing=False so train_on_responses_only can mask prompt tokens; TRL's packer
        # concatenates examples and the two are mutually exclusive. This is a deliberate
        # QUALITY trade, not a correctness fix - Gemma 4 uses sliding-window + full
        # attention (no recurrent state), so packing is safe here, unlike on Qwen3.8's
        # gated-delta layers. For persona training the loss signal is worth far more than
        # the token-utilisation gain.
        packing=False,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=5,
        num_train_epochs=TARGET_EPOCHS,
        learning_rate=LEARNING_RATE,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir=OUTPUT_DIR_ADAPTERS,
        report_to="none",
    ),
)

effective_batch_size = BATCH_SIZE * GRAD_ACCUM
print(f"✓ Trainer configured")
print(f"  Effective batch size: {BATCH_SIZE} × {GRAD_ACCUM} = {effective_batch_size}")
print(f"  Epochs: {TARGET_EPOCHS}")
print(f"  LR: {LEARNING_RATE}")
print(f"  Packing: enabled")
print(f"  Dataset: {len(dataset)} examples")


# ===================== TRAIN ON RESPONSES ONLY =====================
# Mask prompt tokens to -100 so loss is computed ONLY on assistant turns.
# Without this the loss averages in the system prompt and the user turns - tokens the
# base model already predicts near-perfectly and that repeat across every row. They pin
# the average near their own ~0 loss and bury the signal from the response tokens.
# Persona datasets are the worst case: the system prompt is ~1,600 characters and is
# byte-identical across thousands of examples.
#
# Both marker args are left as None so Unsloth auto-detects them from this model's chat
# template and prints what it found - no hand-written marker strings to get wrong.
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(trainer)

# Verify the mask. If the markers do not match the template every label becomes -100,
# the run trains on nothing, and it fails silently for the entire duration.
import numpy as np

_probe = trainer.train_dataset[:8]["labels"]
_kept = sum(int((np.array(x) != -100).sum()) for x in _probe)
_total = sum(len(x) for x in _probe)
if _kept == 0:
    raise RuntimeError(
        "train_on_responses_only masked EVERY token - the instruction/response markers "
        "did not match the chat template. Do not start training."
    )
print(f"Response masking OK: {_kept:,}/{_total:,} label tokens kept "
      f"({100 * _kept / _total:.1f}%) across 8 sample sequences")


## 9. Train

In [ ]:
# Start training
trainer.train()

## 10. Save LoRA Adapters

Save the trained LoRA adapters. These can be loaded on compatible Gemma 4 E4B models with PEFT or served via vLLM.

In [ ]:

from pathlib import Path
import json

Path(LORA_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# Save LoRA adapters + tokenizer
print(f"Saving LoRA adapters to {LORA_OUTPUT_DIR}...")
model.save_pretrained(LORA_OUTPUT_DIR)
(processor or tokenizer).save_pretrained(LORA_OUTPUT_DIR)

# Save system prompts alongside adapters for inference use
prompts_path = f"{LORA_OUTPUT_DIR}/persona_system_prompts.json"
with open(prompts_path, "w") as f:
    json.dump(persona_system_prompts, f, indent=2)

print(f"\n✓ LoRA adapters saved!")
print(f"  Adapters:       {LORA_OUTPUT_DIR}")
print(f"  System prompts: {prompts_path} ({len(persona_system_prompts)} personas)")
print(f"\n  At inference, load prompts with:")
print(f'    with open("{prompts_path}") as f:')
print(f'        prompts = json.load(f)')
print(f'    system_msg = prompts["amos"]  # or any persona key')


## 11. Test Inference

Quick smoke test with a few personas using their extracted system prompts. Each persona should respond in its distinctive voice.


In [ ]:
from transformers import TextStreamer

FastLanguageModel.for_inference(model)

# Pick up to 4 personas to test
test_personas = list(persona_system_prompts.keys())[:4]

print(f"INFERENCE TEST — {len(test_personas)} PERSONAS\n")

for persona_key in test_personas:
    system_prompt = persona_system_prompts[persona_key]

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": TEST_PROMPT},
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    # Gemma 4's tokenizer is a multimodal Processor; calling it positionally
    # binds the string to its first param (`images`), not `text`, so it must
    # be passed as a keyword argument.
    inputs = tokenizer(text=text, return_tensors="pt").to(model.device)

    print(f"{'='*60}")
    print(f"  PERSONA: {persona_key.upper()}")
    print(f"  Q: {TEST_PROMPT}")
    print(f"  A: ", end="")

    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.7,
        top_p=0.95,
        top_k=64,
        do_sample=True,
        streamer=TextStreamer(tokenizer, skip_prompt=True),
    )
    print()

del inputs, outputs


## 12. Verify Adapter (Reload from Disk)

Final validation: load the adapter cold from disk to confirm it's self-contained and portable.


In [ ]:
# Clean up training model
import gc, torch
del model, tokenizer, trainer, dataset
gc.collect()
torch.cuda.empty_cache()

print("✓ Cleared training model from memory")
print(f"  Loading adapter from: {LORA_OUTPUT_DIR}")

# Reload from disk
model2, tokenizer2 = FastLanguageModel.from_pretrained(
    model_name=LORA_OUTPUT_DIR,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model2)

# Reload saved system prompts
import json
with open(f"{LORA_OUTPUT_DIR}/persona_system_prompts.json") as f:
    reloaded_prompts = json.load(f)

# Test with first persona
test_key = list(reloaded_prompts.keys())[0]
test_prompt_text = reloaded_prompts[test_key]

messages = [
    {"role": "system", "content": test_prompt_text},
    {"role": "user", "content": TEST_PROMPT},
]

text = tokenizer2.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
inputs = tokenizer2(text, return_tensors="pt").to(model2.device)

outputs = model2.generate(
    **inputs,
    max_new_tokens=256,
    temperature=0.7,
    top_p=0.95,
    top_k=64,
    do_sample=True,
)

response = tokenizer2.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

print(f"\nADAPTER RELOAD TEST (persona: {test_key}):")
print(f"  Q: {TEST_PROMPT}")
print(f"  A: {response[:500]}")
print(f"\n✓ Adapter loads cleanly from disk. Ready for deployment via vLLM.")

# List adapter files
print(f"\nAdapter contents:")
for p in sorted(Path(LORA_OUTPUT_DIR).iterdir()):
    size_mb = p.stat().st_size / 1024 / 1024
    print(f"  {p.name:40s} {size_mb:>8.1f} MB")

del model2, tokenizer2, inputs, outputs
gc.collect()
torch.cuda.empty_cache()


## 13. Export Merged Model to GGUF (Multi-Format Export)

Merge the LoRA adapter into the base model and export to GGUF so it can
run anywhere llama.cpp / Ollama / LM Studio / iOS GGUF runners are supported
(e.g. iPhone via apps like LLMFarm, PocketPal, Private LLM).

- `q4_k_m` is the recommended quant for phones (best size/quality tradeoff).
- `q5_k_m` and `q8_0` are also produced for higher-quality desktop use.
- Requires Unsloth's bundled `llama.cpp` build (downloaded automatically on
  first call to `save_pretrained_gguf`).

**Note:** Gemma 4 GGUF export requires a recent llama.cpp version that
supports the Gemma 4 architecture. If conversion fails, update llama.cpp
or wait for upstream support to land.

In [ ]:
# Reload adapter fresh so we export from a clean state
import gc, torch
from pathlib import Path
from unsloth import FastLanguageModel

try:
    del model2, tokenizer2
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()

GGUF_OUTPUT_DIR = f"{OUTPUT_BASE_DIR}/gguf"
Path(GGUF_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# Load base + adapter; Unsloth's GGUF exporter will merge before conversion
model_gguf, tokenizer_gguf = FastLanguageModel.from_pretrained(
    model_name=LORA_OUTPUT_DIR,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,   # full-precision merge for accurate GGUF quantization
)

# Pass ALL quant methods in one call so the LoRA→FP16 merge happens ONCE
# and llama.cpp quantizes from that single merged file. Otherwise the loop
# re-merges and re-writes the FP16 GGUF for every quant level.
QUANT_METHODS = ["q4_k_m", "q5_k_m", "q8_0"]

print(f"Exporting GGUF (single merge → {len(QUANT_METHODS)} quants)...")
model_gguf.save_pretrained_gguf(
    GGUF_OUTPUT_DIR,
    tokenizer_gguf,
    quantization_method=QUANT_METHODS,
)

print(f"\n\u2713 GGUF export complete: {GGUF_OUTPUT_DIR}")
for f in sorted(Path(GGUF_OUTPUT_DIR).glob("*.gguf")):
    size_mb = f.stat().st_size / 1024 / 1024
    print(f"  {f.name:60s} {size_mb:>8.1f} MB")

print("\nGGUF deployment (llama.cpp / Ollama / LM Studio):")
print("  1. Deploy the q4_k_m .gguf file to your target platform (local/remote inference).")
print("  2. Load in llama.cpp-compatible tooling (Ollama, LM Studio, llamafile, etc.).")
print("  3. Configure Gemma chat template; set context length <= MAX_SEQ_LENGTH."  )

del model_gguf, tokenizer_gguf
gc.collect()
torch.cuda.empty_cache()
